# LeetCode 730 — Count Different Palindromic Subsequences bằng Python

- Link: https://leetcode.com/problems/count-different-palindromic-subsequences/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động trên đoạn (interval DP), Chuỗi, Đếm phần tử **phân biệt**, Đệ quy có nhớ (memoization), Modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countPalindromicSubsequences(self, s: str) -> int:
```

## 1. Giới thiệu & mục tiêu học

Đếm dãy con đối xứng (palindrome) thì nhiều người đã gặp. Cái khó của bài này nằm ở chữ **"khác nhau" (different)**: hai dãy con lấy ở vị trí khác nhau nhưng tạo ra **cùng một chuỗi** chỉ được đếm **một lần**. Đếm "phân biệt" là một trong những dạng DP khó nhất vì rất dễ đếm trùng.

May mắn: chuỗi chỉ gồm 4 ký tự `a, b, c, d`. Chính điều này mở ra một lời giải rất đẹp: **phân loại palindrome theo ký tự ngoài cùng**.

Sau notebook này bạn sẽ:

1. Viết brute force liệt kê mọi dãy con bằng `set` để làm trọng tài.
2. Hiểu ý tưởng "**chọn ký tự bọc ngoài, lấy lần xuất hiện đầu tiên và cuối cùng**" để không đếm trùng.
3. Viết đệ quy có nhớ `f(i, j)` với `O(4·n^2)` trạng thái-chuyển.
4. Viết DP đoạn kinh điển `dp[i][j]` với ba trường hợp khi `s[i] == s[j]`, và DP 3 chiều `dp[c][i][j]`.
5. Xử lý modulo `10^9 + 7` đúng khi có phép **trừ**.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `s`. Trả về số lượng **dãy con đối xứng khác rỗng, phân biệt** của `s`, lấy modulo `10^9 + 7`.

- Dãy con: xoá 0 hoặc nhiều ký tự, giữ thứ tự phần còn lại.
- Đối xứng: đọc xuôi và ngược giống nhau.
- Hai dãy con được coi là khác nhau nếu **chuỗi** của chúng khác nhau (khác ở ít nhất một vị trí).

**Ví dụ 1**

```text
Input:  s = "bccb"
Output: 6
Giải thích: 'b', 'c', 'bb', 'cc', 'bcb', 'bccb'.
'bcb' xuất hiện hai lần (dùng c thứ nhất hoặc c thứ hai) nhưng chỉ đếm một lần.
```

**Ví dụ 2**

```text
Input:  s = "abcdabcdabcdabcdabcdabcdabcdabcddcbadcbadcbadcbadcbadcbadcbadcba"
Output: 104860361
Giải thích: có 3104860382 dãy con đối xứng phân biệt; 3104860382 mod (10^9+7) = 104860361.
```

**Ràng buộc**

- `1 <= s.length <= 1000`
- `s[i]` là một trong `'a'`, `'b'`, `'c'`, `'d'`.

In [1]:
MOD = 10**9 + 7
EXAMPLES = [
    ("bccb", 6),
    ("abcdabcdabcdabcdabcdabcdabcdabcddcbadcbadcbadcbadcbadcbadcbadcba", 104860361),
]
for s, e in EXAMPLES:
    print(len(s), s[:20] + ("..." if len(s) > 20 else ""), "->", e)
print(3104860382 % MOD)

4 bccb -> 6
64 abcdabcdabcdabcdabcd... -> 104860361
104860361


## 3. Kiến thức nền

### 3.1. Dãy con và `itertools.combinations`

Một dãy con độ dài `k` tương ứng với một tập `k` chỉ số tăng dần. `combinations(range(n), k)` sinh mọi tập như vậy.

In [2]:
from itertools import combinations
s = "bccb"
subs = ["".join(s[i] for i in idx) for k in range(1, len(s) + 1) for idx in combinations(range(len(s)), k)]
print(len(subs), subs)

15 ['b', 'c', 'c', 'b', 'bc', 'bc', 'bb', 'cc', 'cb', 'cb', 'bcc', 'bcb', 'bcb', 'ccb', 'bccb']


### 3.2. `set` để loại trùng và kiểm tra palindrome bằng `t == t[::-1]`

In [3]:
pal = {t for t in subs if t == t[::-1]}
print(sorted(pal, key=lambda t: (len(t), t)), len(pal))

['b', 'c', 'bb', 'cc', 'bcb', 'bccb'] 6


### 3.3. Modulo khi có phép trừ

Trong Python, `%` luôn trả về số **không âm** khi số chia dương, kể cả khi số bị chia âm: `(-3) % 7 == 4`. Nhờ vậy `(a - b) % MOD` luôn đúng. (Trong C++/Java phải tự cộng thêm `MOD`.)

In [4]:
a, b = 5, 9
print((a - b) % MOD, (a - b + MOD) % MOD)

1000000003 1000000003


### 3.4. `functools.lru_cache` — đệ quy có nhớ

Đặt `@lru_cache(None)` trên hàm thuần (chỉ phụ thuộc tham số) để Python tự lưu kết quả. Nhớ tăng `sys.setrecursionlimit` khi độ sâu có thể lớn.

In [5]:
from functools import lru_cache
import sys
sys.setrecursionlimit(10000)

@lru_cache(None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(200))

280571172992510140037611932413038677189525


### 3.5. Bảng "lần xuất hiện kế tiếp / trước đó"

`nxt[i][c]` = chỉ số nhỏ nhất `≥ i` có ký tự `c` (hoặc `n` nếu không có); `prv[j][c]` = chỉ số lớn nhất `≤ j` có ký tự `c` (hoặc `-1`). Tính trong `O(4n)` và tra cứu `O(1)`.

In [6]:
def build_tables(s):
    n = len(s)
    nxt = [[n] * 4 for _ in range(n + 1)]
    for i in range(n - 1, -1, -1):
        nxt[i] = nxt[i + 1][:]
        nxt[i][ord(s[i]) - 97] = i
    prv = [[-1] * 4 for _ in range(n)]
    for j in range(n):
        if j:
            prv[j] = prv[j - 1][:]
        prv[j][ord(s[j]) - 97] = j
    return nxt, prv

nxt, prv = build_tables("bccb")
print("nxt[1] (a,b,c,d) =", nxt[1], " prv[2] =", prv[2])

nxt[1] (a,b,c,d) = [4, 3, 1, 4]  prv[2] = [-1, 0, 2, -1]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Phân loại theo ký tự ngoài cùng

Mọi palindrome khác rỗng có dạng:

- **một ký tự** `x`, hoặc
- `x + (palindrome ở giữa, có thể rỗng) + x`.

Vì chỉ có 4 ký tự, ta chia tập các palindrome phân biệt của đoạn `s[i..j]` thành 4 nhóm theo ký tự ngoài cùng `x ∈ {a,b,c,d}`. Các nhóm **rời nhau** → cộng lại không trùng.

### 4.2. Đếm nhóm `x` mà không trùng

Gọi `l` = vị trí **đầu tiên** của `x` trong `[i, j]`, `r` = vị trí **cuối cùng**.

- Không có `x`: nhóm rỗng → 0.
- `l == r`: chỉ có palindrome `"x"` → 1.
- `l < r`: có `"x"`, `"xx"`, và `"x" + P + "x"` với mọi palindrome phân biệt `P` của `s[l+1..r-1]` → `2 + f(l+1, r-1)`.

Tại sao dùng `l` đầu tiên và `r` cuối cùng? Vì nếu một chuỗi `xPx` xuất hiện ở bất kỳ đâu trong `[i, j]`, thì `P` cũng xuất hiện trong khoảng rộng nhất `(l, r)`. Chọn khoảng rộng nhất **gom tất cả** các cách thành một, mỗi chuỗi `P` phân biệt cho đúng một chuỗi `xPx` phân biệt.

### 4.3. Tính tay `s = "bccb"`

`f(0,3)`:

- `x = a`: không có → 0.
- `x = b`: `l = 0, r = 3` → `2 + f(1,2)`. Mà `f(1,2)` (đoạn `"cc"`): chỉ nhóm `c`: `l=1, r=2` → `2 + f(2,1) = 2 + 0 = 2`. Vậy nhóm b = `2 + 2 = 4` (`b, bb, bcb, bccb`).
- `x = c`: `l = 1, r = 2` → `2 + f(2,1) = 2` (`c, cc`).
- `x = d`: 0.

Tổng `= 4 + 2 = 6`. ✔

In [7]:
print("nhóm b:", ["b", "bb", "bcb", "bccb"], " nhóm c:", ["c", "cc"], " tổng", 4 + 2)

nhóm b: ['b', 'bb', 'bcb', 'bccb']  nhóm c: ['c', 'cc']  tổng 6


## 5. Cách 1 — Brute Force: liệt kê mọi dãy con

**Ý tưởng.** Sinh `2^n - 1` dãy con, giữ những cái là palindrome trong một `set`, trả về kích thước `set`.

In [8]:
def count_brute(s):
    n = len(s)
    seen = set()
    for mask in range(1, 1 << n):
        t = "".join(s[i] for i in range(n) if mask >> i & 1)
        if t == t[::-1]:
            seen.add(t)
    return len(seen) % MOD

print(count_brute("bccb"))
print(count_brute("abcdcba"))

6
22


**Độ phức tạp.** `O(2^n · n)` thời gian, bộ nhớ `O(số palindrome phân biệt · n)`. `n = 20` đã là 20 triệu thao tác; `n = 1000` là bất khả.

**Nút thắt.** Ta sinh từng dãy con **theo vị trí**, nên cùng một chuỗi bị sinh rất nhiều lần rồi mới loại trùng. Cần một cách đếm mà **mỗi chuỗi phân biệt sinh ra đúng một lần** — đó là mục 4.2.

## 6. Bước đột phá

**Insight.** Với mỗi ký tự ngoài cùng `x`, "đại diện chuẩn" của chuỗi `xPx` trong đoạn `[i, j]` là cách dùng **`x` đầu tiên** và **`x` cuối cùng**. Mỗi chuỗi phân biệt ↔ đúng một đại diện chuẩn → đếm không trùng, không sót.

**Công thức (đệ quy theo ký tự):**

```text
f(i, j) = Σ_{x ∈ abcd} g(x, i, j)
g(x, i, j) = 0                      nếu x không có trong [i, j]
           = 1                      nếu x xuất hiện đúng 1 lần
           = 2 + f(l+1, r-1)        nếu l < r  (l = nxt[i][x], r = prv[j][x])
f(i, j) = 0 khi i > j
```

Số trạng thái `f(i, j)` là `O(n^2)`, mỗi trạng thái tốn `O(4)` nhờ bảng `nxt/prv` → `O(n^2)`.

**Quy tắc nếu… thì… (DP đoạn kinh điển — Cách 3)** khi so sánh hai đầu:

- Nếu `s[i] != s[j]`: bao hàm – loại trừ: `dp[i][j] = dp[i+1][j] + dp[i][j-1] - dp[i+1][j-1]`.
- Nếu `s[i] == s[j] = x`: tìm `low`, `high` là `x` đầu tiên/cuối cùng **bên trong** `(i, j)`:
  - không có `x` bên trong: `2·dp[i+1][j-1] + 2` (thêm `"x"` và `"xx"`),
  - đúng một `x` bên trong: `2·dp[i+1][j-1] + 1` (chỉ thêm `"xx"`, vì `"x"` đã có),
  - ≥ 2 `x` bên trong: `2·dp[i+1][j-1] - dp[low+1][high-1]` (trừ phần bị đếm hai lần).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ theo ký tự ngoài cùng

**Giải thích.** Cài trực tiếp công thức ở mục 6. Dùng `lru_cache` trên `f(i, j)`.

In [9]:
def count_memo(s):
    n = len(s)
    nxt, prv = build_tables(s)

    @lru_cache(None)
    def f(i, j):
        if i > j:
            return 0
        total = 0
        for x in range(4):
            l, r = nxt[i][x], prv[j][x]
            if l > j:                 # không có x trong [i, j]
                continue
            if l == r:
                total += 1
            else:
                total += 2 + f(l + 1, r - 1)
        return total % MOD

    return f(0, n - 1)

for s, e in EXAMPLES:
    print(count_memo(s), count_memo(s) == e)

6 True
104860361 True


**Code nộp LeetCode (Cách 2).** Độ sâu đệ quy ≤ `n/2 = 500`.

```python
import sys
from functools import lru_cache

class Solution:
    def countPalindromicSubsequences(self, s: str) -> int:
        sys.setrecursionlimit(10000)
        MOD = 10**9 + 7
        n = len(s)
        nxt = [[n] * 4 for _ in range(n + 1)]
        for i in range(n - 1, -1, -1):
            nxt[i] = nxt[i + 1][:]
            nxt[i][ord(s[i]) - 97] = i
        prv = [[-1] * 4 for _ in range(n)]
        for j in range(n):
            if j:
                prv[j] = prv[j - 1][:]
            prv[j][ord(s[j]) - 97] = j

        @lru_cache(None)
        def f(i, j):
            if i > j:
                return 0
            total = 0
            for x in range(4):
                l, r = nxt[i][x], prv[j][x]
                if l > j:
                    continue
                total += 1 if l == r else 2 + f(l + 1, r - 1)
            return total % MOD

        return f(0, n - 1)
```

**Độ phức tạp.** Chỉ những cặp `(l+1, r-1)` thực sự được gọi mới tính — thường ít hơn nhiều so với `n^2`. Trường hợp xấu `O(4·n^2)` time, `O(n^2)` space.

### 7.2. Cách 3 — DP đoạn kinh điển `dp[i][j]`

**Giải thích.** `dp[i][j]` = số palindrome phân biệt trong `s[i..j]`. Tính theo độ dài đoạn tăng dần (hoặc `i` giảm dần, `j` tăng dần).

Giải thích 3 trường hợp khi `s[i] == s[j] = x`:

- Mọi palindrome `P` của `s[i+1..j-1]` cho ra hai palindrome của `s[i..j]`: `P` và `xPx`. → `2·dp[i+1][j-1]`.
- Nếu **không** có `x` bên trong: `"x"` và `"xx"` là mới → `+2`.
- Nếu có **đúng một** `x` bên trong: `"x"` đã được đếm (là một `P`), chỉ `"xx"` mới → `+1`.
- Nếu có **≥ 2** `x` bên trong (vị trí `low < high`): mỗi chuỗi `xQx` với `Q` là palindrome của `s[low+1..high-1]` bị đếm **hai lần** — một lần như một `P` (nó đã nằm trong `dp[i+1][j-1]`) và một lần như `xPx` với `P = Q`. Trừ đi `dp[low+1][high-1]`. Còn `"x"`, `"xx"` đều đã có bên trong → không cộng thêm (đếm chi tiết từng phần tử xem mục 10).

In [10]:
def count_interval_dp(s):
    n = len(s)
    nxt, prv = build_tables(s)
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, n):
            if s[i] != s[j]:
                dp[i][j] = dp[i + 1][j] + dp[i][j - 1] - dp[i + 1][j - 1]
            else:
                x = ord(s[i]) - 97
                low = nxt[i + 1][x]           # x đầu tiên bên trong
                high = prv[j - 1][x]          # x cuối cùng bên trong
                inner = dp[i + 1][j - 1] if i + 1 <= j - 1 else 0
                if low > high:                # không có x bên trong
                    dp[i][j] = 2 * inner + 2
                elif low == high:             # đúng một x
                    dp[i][j] = 2 * inner + 1
                else:
                    sub = dp[low + 1][high - 1] if low + 1 <= high - 1 else 0
                    dp[i][j] = 2 * inner - sub
            dp[i][j] %= MOD
    return dp[0][n - 1]

for s, e in EXAMPLES:
    print(count_interval_dp(s), count_interval_dp(s) == e)

6 True
104860361 True


> Chú ý trường hợp `j == i + 1`: `dp[i+1][j-1] = dp[i+1][i]` là đoạn rỗng → phải là `0`. Vì bảng khởi tạo toàn `0` và ta chỉ ghi `dp[i][j]` với `i ≤ j`, ô `dp[i+1][i]` vẫn là `0` — nhưng viết rõ `if i + 1 <= j - 1` giúp đọc dễ hơn. Cũng chú ý khi `j == i+1`, `nxt[i+1][x]` có thể là `j` (chính `s[j]`), lớn hơn `prv[j-1][x]` → rơi vào nhánh "không có x bên trong" — đúng.

**Code nộp LeetCode (Cách 3).**

```python
class Solution:
    def countPalindromicSubsequences(self, s: str) -> int:
        MOD = 10**9 + 7
        n = len(s)
        nxt = [[n] * 4 for _ in range(n + 1)]
        for i in range(n - 1, -1, -1):
            nxt[i] = nxt[i + 1][:]
            nxt[i][ord(s[i]) - 97] = i
        prv = [[-1] * 4 for _ in range(n)]
        for j in range(n):
            if j:
                prv[j] = prv[j - 1][:]
            prv[j][ord(s[j]) - 97] = j
        dp = [[0] * n for _ in range(n)]
        for i in range(n - 1, -1, -1):
            dp[i][i] = 1
            row, down = dp[i], dp[i + 1] if i + 1 < n else None
            for j in range(i + 1, n):
                if s[i] != s[j]:
                    row[j] = (down[j] + row[j - 1] - down[j - 1]) % MOD
                else:
                    x = ord(s[i]) - 97
                    low, high = nxt[i + 1][x], prv[j - 1][x]
                    inner = down[j - 1]
                    if low > high:
                        row[j] = (2 * inner + 2) % MOD
                    elif low == high:
                        row[j] = (2 * inner + 1) % MOD
                    else:
                        row[j] = (2 * inner - dp[low + 1][high - 1]) % MOD
        return dp[0][n - 1]
```

(Trong bản nộp, `down[j-1]` với `j = i+1` là `dp[i+1][i] = 0`; `dp[low+1][high-1]` với `low+1 > high-1` là ô dưới đường chéo, cũng bằng `0`.)

**Độ phức tạp.** `O(n^2)` time, `O(n^2)` space (≈ `10^6` ô với `n = 1000`).

### 7.3. Cách 4 — DP 3 chiều `dp[x][i][j]` (bottom-up theo ký tự)

**Giải thích.** `dp[x][i][j]` = số palindrome phân biệt trong `s[i..j]` **bắt đầu và kết thúc bằng `x`**. Đây là bản bottom-up của Cách 2, nhưng không cần bảng `nxt/prv`:

- Nếu `s[i] != x`: `dp[x][i][j] = dp[x][i+1][j]` (bỏ đầu trái).
- Nếu `s[j] != x`: `dp[x][i][j] = dp[x][i][j-1]` (bỏ đầu phải).
- Nếu `s[i] == s[j] == x`:
  - `i == j`: 1 (`"x"`),
  - ngược lại: `2 + Σ_y dp[y][i+1][j-1]` (`"x"`, `"xx"`, và `xPx`).

Đáp án `Σ_x dp[x][0][n-1]`.

In [11]:
def count_3d(s):
    n = len(s)
    dp = [[[0] * n for _ in range(n)] for _ in range(4)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            for x in range(4):
                ch = chr(97 + x)
                if s[i] != ch:
                    dp[x][i][j] = dp[x][i + 1][j] if i + 1 <= j else 0
                elif s[j] != ch:
                    dp[x][i][j] = dp[x][i][j - 1]
                elif i == j:
                    dp[x][i][j] = 1
                else:
                    inner = sum(dp[y][i + 1][j - 1] for y in range(4)) if i + 1 <= j - 1 else 0
                    dp[x][i][j] = (2 + inner) % MOD
    return sum(dp[x][0][n - 1] for x in range(4)) % MOD

for s, e in EXAMPLES:
    print(count_3d(s), count_3d(s) == e)

6 True
104860361 True


**Code nộp LeetCode (Cách 4).**

```python
class Solution:
    def countPalindromicSubsequences(self, s: str) -> int:
        MOD = 10**9 + 7
        n = len(s)
        dp = [[[0] * n for _ in range(n)] for _ in range(4)]
        for i in range(n - 1, -1, -1):
            for j in range(i, n):
                for x in range(4):
                    ch = 'abcd'[x]
                    if s[i] != ch:
                        dp[x][i][j] = dp[x][i + 1][j] if i + 1 <= j else 0
                    elif s[j] != ch:
                        dp[x][i][j] = dp[x][i][j - 1]
                    elif i == j:
                        dp[x][i][j] = 1
                    else:
                        inner = 0
                        if i + 1 <= j - 1:
                            inner = dp[0][i+1][j-1] + dp[1][i+1][j-1] + dp[2][i+1][j-1] + dp[3][i+1][j-1]
                        dp[x][i][j] = (2 + inner) % MOD
        return sum(dp[x][0][n - 1] for x in range(4)) % MOD
```

**Độ phức tạp.** `O(4·n^2)` time, `O(4·n^2)` space — gấp 4 lần Cách 3, trong Python có thể chậm với `n = 1000`, nhưng ý tưởng rất trong sáng: **không có phép trừ**, nên không lo modulo âm.

## 8. TRACE từng bước

### 8.1. Bảng `dp[i][j]` (Cách 3) cho `s = "bccb"`

Hàng = `i`, cột = `j`:

| i\j | 0 (b) | 1 (c) | 2 (c) | 3 (b) |
|---|---|---|---|---|
| 0 | 1 | 2 | 3 | **6** |
| 1 | · | 1 | 2 | 3 |
| 2 | · | · | 1 | 2 |
| 3 | · | · | · | 1 |

- `dp[1][2]` (`"cc"`): `s[1]==s[2]='c'`, không có `c` bên trong → `2·0 + 2 = 2`.
- `dp[0][2]` (`"bcc"`): khác nhau → `dp[1][2] + dp[0][1] - dp[1][1] = 2 + 2 - 1 = 3` (`b, c, cc`).
- `dp[0][3]`: `s[0]==s[3]='b'`, không có `b` bên trong → `2·dp[1][2] + 2 = 6`.

In [12]:
def trace_interval_dp(s):
    n = len(s)
    nxt, prv = build_tables(s)
    dp = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, n):
            inner = dp[i + 1][j - 1] if i + 1 <= j - 1 else 0
            if s[i] != s[j]:
                dp[i][j] = dp[i + 1][j] + dp[i][j - 1] - inner
                why = f"khác: {dp[i+1][j]} + {dp[i][j-1]} - {inner}"
            else:
                x = ord(s[i]) - 97
                low, high = nxt[i + 1][x], prv[j - 1][x]
                if low > high:
                    dp[i][j] = 2 * inner + 2; why = f"giống, 0 '{s[i]}' trong: 2*{inner}+2"
                elif low == high:
                    dp[i][j] = 2 * inner + 1; why = f"giống, 1 '{s[i]}' trong: 2*{inner}+1"
                else:
                    sub = dp[low + 1][high - 1] if low + 1 <= high - 1 else 0
                    dp[i][j] = 2 * inner - sub; why = f"giống, >=2 '{s[i]}' (low={low},high={high}): 2*{inner}-{sub}"
            print(f"dp[{i}][{j}] = {dp[i][j]:<3} ({s[i:j+1]!r:8}) {why}")
    return dp[0][n - 1]

print("Kết quả:", trace_interval_dp("bccb"))

dp[2][3] = 2   ('cb'    ) khác: 1 + 1 - 0
dp[1][2] = 2   ('cc'    ) giống, 0 'c' trong: 2*0+2
dp[1][3] = 3   ('ccb'   ) khác: 2 + 2 - 1
dp[0][1] = 2   ('bc'    ) khác: 1 + 1 - 0
dp[0][2] = 3   ('bcc'   ) khác: 2 + 2 - 1
dp[0][3] = 6   ('bccb'  ) giống, 0 'b' trong: 2*2+2
Kết quả: 6


### 8.2. Trace trường hợp "≥ 2 ký tự giống bên trong": `s = "aaaa"`

Palindrome phân biệt của `"aaaa"`: `a, aa, aaa, aaaa` → 4.

`dp[0][3]`: `s[0]==s[3]='a'`, bên trong `"aa"` có `low=1, high=2` → `2·dp[1][2] - dp[2][1] = 2·2 - 0 = 4`. ✔ (Nếu quên trừ, ta sẽ ra `2·2 + 2 = 6` — sai.)

In [13]:
print("Kết quả:", trace_interval_dp("aaaa"))

dp[2][3] = 2   ('aa'    ) giống, 0 'a' trong: 2*0+2
dp[1][2] = 2   ('aa'    ) giống, 0 'a' trong: 2*0+2
dp[1][3] = 3   ('aaa'   ) giống, 1 'a' trong: 2*1+1
dp[0][1] = 2   ('aa'    ) giống, 0 'a' trong: 2*0+2
dp[0][2] = 3   ('aaa'   ) giống, 1 'a' trong: 2*1+1
dp[0][3] = 4   ('aaaa'  ) giống, >=2 'a' (low=1,high=2): 2*2-0
Kết quả: 4


### 8.3. Trace Cách 2 (đệ quy theo ký tự) — in cây lời gọi

In [14]:
def trace_memo(s):
    n = len(s)
    nxt, prv = build_tables(s)
    memo = {}
    def f(i, j, depth=0):
        if i > j:
            return 0
        if (i, j) in memo:
            print("  " * depth + f"f({i},{j}) = {memo[(i, j)]} (nhớ)")
            return memo[(i, j)]
        print("  " * depth + f"f({i},{j}) đoạn {s[i:j+1]!r}")
        total = 0
        for x in range(4):
            l, r = nxt[i][x], prv[j][x]
            if l > j:
                continue
            if l == r:
                print("  " * depth + f"  '{chr(97+x)}': một lần -> 1")
                total += 1
            else:
                sub = f(l + 1, r - 1, depth + 2)
                print("  " * depth + f"  '{chr(97+x)}': l={l}, r={r} -> 2 + {sub}")
                total += 2 + sub
        memo[(i, j)] = total
        print("  " * depth + f"=> f({i},{j}) = {total}")
        return total
    return f(0, n - 1)

print("Kết quả:", trace_memo("abcab"))

f(0,4) đoạn 'abcab'
    f(1,2) đoạn 'bc'
      'b': một lần -> 1
      'c': một lần -> 1
    => f(1,2) = 2
  'a': l=0, r=3 -> 2 + 2
    f(2,3) đoạn 'ca'
      'a': một lần -> 1
      'c': một lần -> 1
    => f(2,3) = 2
  'b': l=1, r=4 -> 2 + 2
  'c': một lần -> 1
=> f(0,4) = 9
Kết quả: 9


## 9. Code nộp LeetCode

Bản chính: **Cách 3 (DP đoạn)** với bảng `nxt/prv` → `O(n^2)` chắc chắn.

In [15]:
class Solution:
    def countPalindromicSubsequences(self, s: str) -> int:
        MOD = 10**9 + 7
        n = len(s)
        nxt = [[n] * 4 for _ in range(n + 1)]
        for i in range(n - 1, -1, -1):
            nxt[i] = nxt[i + 1][:]
            nxt[i][ord(s[i]) - 97] = i
        prv = [[-1] * 4 for _ in range(n)]
        for j in range(n):
            if j:
                prv[j] = prv[j - 1][:]
            prv[j][ord(s[j]) - 97] = j
        dp = [[0] * n for _ in range(n)]
        for i in range(n - 1, -1, -1):
            dp[i][i] = 1
            row = dp[i]
            down = dp[i + 1] if i + 1 < n else None
            for j in range(i + 1, n):
                if s[i] != s[j]:
                    row[j] = (down[j] + row[j - 1] - down[j - 1]) % MOD
                else:
                    x = ord(s[i]) - 97
                    low, high = nxt[i + 1][x], prv[j - 1][x]
                    inner = down[j - 1]
                    if low > high:
                        row[j] = (2 * inner + 2) % MOD
                    elif low == high:
                        row[j] = (2 * inner + 1) % MOD
                    else:
                        row[j] = (2 * inner - dp[low + 1][high - 1]) % MOD
        return dp[0][n - 1]

for s, e in EXAMPLES:
    assert Solution().countPalindromicSubsequences(s) == e
print("Solution OK trên ví dụ đề")

Solution OK trên ví dụ đề


## 10. Vì sao thuật toán đúng

**Cách 2 (song ánh).** Gọi `Pal(i, j)` là tập palindrome phân biệt của `s[i..j]`. Chia theo ký tự ngoài cùng `x` (rời nhau). Với nhóm `x`, gọi `l, r` là `x` đầu/cuối trong `[i, j]`:

- `"x"` ∈ nhóm ⇔ `x` có mặt.
- `"xx"` ∈ nhóm ⇔ `x` có ít nhất 2 lần ⇔ `l < r`.
- `xPx` (`P` khác rỗng) ∈ nhóm ⇔ tồn tại `a < b` với `s[a]=s[b]=x` và `P` là dãy con của `s[a+1..b-1]`. Vì `l ≤ a`, `b ≤ r`, điều này ⇔ `P` là dãy con của `s[l+1..r-1]` ⇔ `P ∈ Pal(l+1, r-1)`.

Ánh xạ `P ↦ xPx` là song ánh giữa `Pal(l+1, r-1)` và phần "độ dài ≥ 3" của nhóm. Vậy nhóm có `2 + |Pal(l+1, r-1)|` phần tử khi `l < r`. ∎

**Cách 3.**

- `s[i] ≠ s[j]`: không palindrome nào dùng **cả** hai đầu (ký tự ngoài cùng phải giống nhau), nên `Pal(i,j) = Pal(i+1,j) ∪ Pal(i,j-1)`; giao của chúng là `Pal(i+1,j-1)` → bao hàm – loại trừ.
- `s[i] = s[j] = x`: `Pal(i,j) = Pal(i+1,j-1) ∪ {xPx : P ∈ Pal(i+1,j-1)} ∪ {"x","xx"}`. Phần giao giữa hai tập đầu là các chuỗi `xQx` đã có trong `Pal(i+1,j-1)` — theo lập luận như Cách 2, đó đúng là `{xQx : Q ∈ Pal(low+1, high-1)} ∪ {"xx"}`, và `"x"` cũng đã có nếu có `x` bên trong. Xét 3 trường hợp số `x` bên trong (0, 1, ≥2) cho ra đúng 3 công thức.

Trường hợp ≥ 2: giao gồm `|Pal(low+1,high-1)|` chuỗi `xQx` và chuỗi `"xx"`; thêm `"x"`, `"xx"` không mới. Tổng `= 2·inner - (sub + 1) + 1 = 2·inner - sub`. ✔

**Modulo.** Phép trừ có thể làm giá trị (sau mod) âm tạm thời; `%` của Python đưa về `[0, MOD)`, và đẳng thức đồng dư vẫn giữ.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(2^n · n)` | `O(#pal · n)` |
| 2. Memo theo ký tự | `O(4·n^2)` xấu nhất (thường ít hơn) | `O(n^2)` |
| 3. DP đoạn + nxt/prv | `O(n^2)` | `O(n^2)` |
| 3'. DP đoạn quét tìm low/high | `O(n^3)` xấu nhất | `O(n^2)` |
| 4. DP 3 chiều | `O(4·n^2)` | `O(4·n^2)` |

Với `n = 1000`: `n^2 = 10^6` ô — Python làm khoảng 1 giây. Dưới đây đo với `n = 400` để notebook chạy nhanh.

In [16]:
import random, time
rng = random.Random(730)
s_big = "".join(rng.choice("abcd") for _ in range(400))
for fn in [count_memo, count_interval_dp, Solution().countPalindromicSubsequences, count_3d]:
    t = time.perf_counter(); r = fn(s_big); t = time.perf_counter() - t
    print(f"{getattr(fn, '__name__', 'Solution'):<32} {t*1000:8.1f} ms  -> {r}")

count_memo                            9.9 ms  -> 499495211
count_interval_dp                    16.1 ms  -> 499495211
countPalindromicSubsequences         10.0 ms  -> 499495211
count_3d                             47.3 ms  -> 499495211


## 12. Bộ test

In [17]:
ALL = [count_brute, count_memo, count_interval_dp, count_3d, Solution().countPalindromicSubsequences]

EDGE = [
    ("a", 1),
    ("aa", 2),
    ("ab", 2),
    ("aaa", 3),
    ("aaaa", 4),
    ("aba", 4),        # a, b, aa, aba
    ("abcd", 4),
    ("abba", 6),       # a, b, aa, bb, aba, abba
]
for fn in ALL:
    for s, e in EDGE:
        assert fn(s) == e, (fn, s, fn(s), e)
for fn in ALL[1:]:
    for s, e in EXAMPLES:
        assert fn(s) == e, (fn, s)
print("Ví dụ đề + edge case OK")

Ví dụ đề + edge case OK


In [18]:
rng = random.Random(7300)
for _ in range(800):
    n = rng.randint(1, 13)
    alpha = "abcd"[:rng.randint(1, 4)]
    s = "".join(rng.choice(alpha) for _ in range(n))
    ref = count_brute(s)
    for fn in ALL[1:]:
        assert fn(s) == ref, (fn, s, fn(s), ref)
print("800 chuỗi ngẫu nhiên: mọi cách khớp brute force")

800 chuỗi ngẫu nhiên: mọi cách khớp brute force


In [19]:
# Chuỗi dài: các cách tối ưu khớp nhau (kiểm tra modulo)
for _ in range(3):
    s = "".join(rng.choice("abcd") for _ in range(300))
    res = {count_memo(s), count_interval_dp(s), Solution().countPalindromicSubsequences(s)}
    assert len(res) == 1, res
print("Chuỗi dài n=300 khớp nhau")

Chuỗi dài n=300 khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Đếm số dãy con palindrome (theo vị trí), không loại trùng

**Code sai:** công thức kinh điển `cnt[i][j] = cnt[i+1][j] + cnt[i][j-1] - cnt[i+1][j-1]` và `+ cnt[i+1][j-1] + 1` khi hai đầu bằng nhau — đó là bài **đếm theo vị trí** (mỗi tập chỉ số là một dãy con riêng).

**Vì sao sai:** đếm theo vị trí thì `"bccb"` có `b` 2 lần, `c` 2 lần, `bcb` 2 lần… → ra 9 thay vì 6.

**Code đúng:** đếm theo chuỗi phân biệt (Cách 2/3).

In [20]:
def count_by_position_wrong(s):
    n = len(s)
    c = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        c[i][i] = 1
        for j in range(i + 1, n):
            if s[i] == s[j]:
                c[i][j] = c[i + 1][j] + c[i][j - 1] + 1
            else:
                c[i][j] = c[i + 1][j] + c[i][j - 1] - (c[i + 1][j - 1] if i + 1 <= j - 1 else 0)
    return c[0][n - 1]

print("sai:", count_by_position_wrong("bccb"), " đúng:", Solution().countPalindromicSubsequences("bccb"))

sai: 9  đúng: 6


### Lỗi 2 — Quên trường hợp "≥ 2 ký tự giống bên trong"

**Code sai:** khi `s[i]==s[j]` luôn dùng `2·inner + 2`.

**Vì sao sai:** với `"aaaa"`, `"aa"` và `"a"` đã có sẵn bên trong, và `a(a)a` = `aaa` bị đếm lặp. Ra 6 thay vì 4.

**Code đúng:** phân 3 trường hợp theo số `x` bên trong.

In [21]:
def two_case_wrong(s):
    n = len(s); dp = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, n):
            inner = dp[i + 1][j - 1] if i + 1 <= j - 1 else 0
            dp[i][j] = 2 * inner + 2 if s[i] == s[j] else dp[i + 1][j] + dp[i][j - 1] - inner
    return dp[0][n - 1]

print("sai:", two_case_wrong("aaaa"), " đúng:", Solution().countPalindromicSubsequences("aaaa"))

sai: 6  đúng: 4


### Lỗi 3 — Modulo âm trong ngôn ngữ khác / quên mod sau phép trừ

**Code sai (tư duy C++/Java):** `dp = (a - b) % MOD` có thể âm; trong Python thì không, nhưng nếu bạn **quên mod** hoàn toàn thì số tăng cực lớn (chậm) và sai so với yêu cầu.

**Code đúng:** `% MOD` sau mỗi phép gán (Python tự đưa về không âm); trong C++/Java viết `((a - b) % MOD + MOD) % MOD`.

In [22]:
a, b = 3, 10**9 + 9   # giả sử sau mod a nhỏ hơn b
print("Python:", (a - b) % MOD, "  kiểu C (cắt về 0):", int(__import__('math').fmod(a - b, MOD)))

Python: 1   kiểu C (cắt về 0): -1000000006


### Lỗi 4 — Dùng `x` gần nhất thay vì `x` đầu/cuối trong Cách 2

**Code sai:** chọn `l` là `x` đầu tiên nhưng `r` là `x` **thứ hai** (gần `l` nhất) → bỏ sót các `P` nằm xa.

**Vì sao sai:** `xPx` cần khoảng **rộng nhất** để chứa mọi `P` có thể.

**Code đúng:** `l = nxt[i][x]`, `r = prv[j][x]`.

In [23]:
def nearest_wrong(s):
    n = len(s); nxt, prv = build_tables(s)
    @lru_cache(None)
    def f(i, j):
        if i > j: return 0
        t = 0
        for x in range(4):
            l = nxt[i][x]
            if l > j: continue
            r = nxt[l + 1][x] if l + 1 <= n else n     # SAI: x kế tiếp gần nhất
            t += 1 if r > j else 2 + f(l + 1, r - 1)
        return t
    return f(0, n - 1)

print("sai:", nearest_wrong("abcba"), " đúng:", Solution().countPalindromicSubsequences("abcba"))

sai: 10  đúng: 10


## 14. Mẹo & nhận dạng pattern

- **"Đếm phân biệt"** → tìm một **đại diện chuẩn** cho mỗi đối tượng (ở đây: dùng ký tự đầu tiên & cuối cùng). Cùng tư tưởng với *Distinct Subsequences II (940)*.
- **Bảng chữ cái nhỏ** (4, 26 ký tự) → có thể **lặp theo ký tự** trong mỗi trạng thái, giữ `O(Σ·n^2)`.
- **Palindrome + dãy con** → DP đoạn `dp[i][j]`, tính theo `i` giảm dần, `j` tăng dần.
- **Bao hàm – loại trừ** khi hai đầu khác nhau: `A ∪ B = A + B - A∩B`.
- Có phép trừ trong DP modulo → kiểm tra dấu (Python an toàn, ngôn ngữ khác thì không).

## 15. Biến thể

1. **Count Palindromic Subsequences theo vị trí** (không phân biệt): công thức `+1` đơn giản (mục 13, Lỗi 1).
2. **Longest Palindromic Subsequence (516)**: DP đoạn tìm độ dài lớn nhất.
3. **Count Palindromic Subsequences độ dài 5 (2484)**: đếm theo vị trí, cố định tâm.
4. **Unique Length-3 Palindromic Subsequences (1930)**: phiên bản dễ của đúng ý tưởng "x đầu tiên + x cuối cùng".
5. **Bảng chữ cái 26 ký tự**: Cách 2 và 3 vẫn đúng, chỉ đổi 4 → 26.

In [24]:
# Biến thể 4 (LeetCode 1930): số palindrome phân biệt độ dài 3
def count_len3(s):
    total = 0
    for x in set(s):
        l, r = s.find(x), s.rfind(x)
        if r - l >= 2:
            total += len(set(s[l + 1:r]))
    return total

def count_len3_brute(s):
    return len({s[i] + s[j] + s[k] for i, j, k in combinations(range(len(s)), 3) if s[i] == s[k]})

for s in ["aabca", "adc", "bbcbaba"]:
    print(s, count_len3(s), count_len3_brute(s))

aabca 3 3
adc 0 0
bbcbaba 4 4


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Liệt kê `2^n` dãy con + `set` | `O(2^n·n)` | lớn | Trọng tài, `n ≤ 15` |
| 2. Memo theo ký tự | `Σ_x (2 + f(l+1, r-1))` | `O(4n^2)` | `O(n^2)` | Dễ chứng minh, dễ nhớ nhất |
| 3. DP đoạn kinh điển | 3 trường hợp khi hai đầu bằng | `O(n^2)` | `O(n^2)` | Bản nộp nhanh |
| 4. DP 3 chiều | `dp[x][i][j]`, không phép trừ | `O(4n^2)` | `O(4n^2)` | Muốn tránh phép trừ modulo |

## 17. Bài tập

**Bài 1 (đoán output).** `countPalindromicSubsequences("abcba")` = ?

**Bài 2 (tính tay).** Dùng Cách 2 tính tay cho `s = "abab"`. Liệt kê các palindrome theo từng nhóm ký tự.

**Bài 3 (điền TODO).** Hoàn thiện `group_count(s, i, j, x, f)` trả về số palindrome trong nhóm ký tự `x` của đoạn `[i, j]` (dùng `str.find`/`str.rfind`).

**Bài 4 (tự viết lại).** Viết lại Cách 3 (DP đoạn) nhưng tìm `low`, `high` bằng vòng `while` (không dùng bảng `nxt/prv`).

**Bài 5 (mini challenge).** Viết `count_distinct_pal_len(s, L)` đếm số palindrome **phân biệt** có độ dài đúng `L` (gợi ý: thêm chiều độ dài vào Cách 2).

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def group_count(s, i, j, x, f):
    # TODO: l = s.find(x, i, j + 1); r = s.rfind(x, i, j + 1)
    #       không có -> 0; l == r -> 1; còn lại -> 2 + f(l + 1, r - 1)
    pass

# Bài 4 — TODO
def interval_dp_while(s):
    # TODO: như Cách 3, nhưng low = i+1; while low <= j-1 and s[low] != s[i]: low += 1 ...
    pass

# Bài 5 — TODO
def count_distinct_pal_len(s, L):
    # TODO: g(i, j, L) = Σ_x [L==1: có x] [L==2: l<r] [L>=3: l<r và g(l+1, r-1, L-2)]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: nhóm `a`: `l=0, r=4` → `2 + f(1,3)`.
- Mức 2: `f(1,3)` của `"bcb"` = nhóm b (`2 + f(2,2) = 3`) + nhóm c (`1`) = 4.
- Mức 3: nhóm a = 6, nhóm b (`l=1,r=3`) = `2 + 1 = 3`, nhóm c = 1 → **10**.

**Bài 2.**
- Mức 1: nhóm a: `l=0, r=2`; nhóm b: `l=1, r=3`.
- Mức 2: nhóm a = `2 + f(1,1) = 3` (`a, aa, aba`); nhóm b = `2 + f(2,2) = 3` (`b, bb, bab`).
- Mức 3: tổng **6**.

**Bài 3.**
- Mức 1: `str.find(x, start, end)` tìm trong `s[start:end]`.
- Mức 2: trả `-1` nếu không thấy.
- Mức 3: chú ý `end = j + 1` (đầu mút phải không bao gồm).

**Bài 4.**
- Mức 1: chỉ thay hai dòng tính `low`, `high`.
- Mức 2: `high = j - 1; while high >= i+1 and s[high] != s[i]: high -= 1`.
- Mức 3: nếu không thấy thì đặt `low = j`, `high = i` để `low > high`.

**Bài 5.**
- Mức 1: palindrome độ dài `L ≥ 3` nhóm `x` ↔ palindrome độ dài `L-2` trong `(l, r)`.
- Mức 2: `L == 1`: 1 nếu có `x`; `L == 2`: 1 nếu `l < r`.
- Mức 3: memo theo `(i, j, L)`; kiểm tra: tổng theo mọi `L` = đáp án bài gốc.

## 19. Đáp án tham khảo

In [26]:
# Bài 1, 2
assert Solution().countPalindromicSubsequences("abcba") == 10
assert Solution().countPalindromicSubsequences("abab") == 6

# Bài 3
def group_count_ans(s, i, j, x, f):
    l, r = s.find(x, i, j + 1), s.rfind(x, i, j + 1)
    if l == -1:
        return 0
    return 1 if l == r else 2 + f(l + 1, r - 1)

def count_via_group(s):
    @lru_cache(None)
    def f(i, j):
        if i > j:
            return 0
        return sum(group_count_ans(s, i, j, x, f) for x in "abcd") % MOD
    return f(0, len(s) - 1)

assert count_via_group("bccb") == 6
print("Bài 1–3 OK")

Bài 1–3 OK


In [27]:
# Bài 4
def interval_dp_while_ans(s):
    n = len(s); dp = [[0] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        dp[i][i] = 1
        for j in range(i + 1, n):
            inner = dp[i + 1][j - 1] if i + 1 <= j - 1 else 0
            if s[i] != s[j]:
                dp[i][j] = dp[i + 1][j] + dp[i][j - 1] - inner
            else:
                low, high = i + 1, j - 1
                while low <= high and s[low] != s[i]:
                    low += 1
                while low <= high and s[high] != s[i]:
                    high -= 1
                if low > high:
                    dp[i][j] = 2 * inner + 2
                elif low == high:
                    dp[i][j] = 2 * inner + 1
                else:
                    sub = dp[low + 1][high - 1] if low + 1 <= high - 1 else 0
                    dp[i][j] = 2 * inner - sub
            dp[i][j] %= MOD
    return dp[0][n - 1]

# Bài 5
def count_distinct_pal_len_ans(s, L):
    @lru_cache(None)
    def g(i, j, L):
        if L <= 0 or i > j:
            return 0
        t = 0
        for x in "abcd":
            l, r = s.find(x, i, j + 1), s.rfind(x, i, j + 1)
            if l == -1:
                continue
            if L == 1:
                t += 1
            elif L == 2:
                t += l < r
            elif l < r:
                t += g(l + 1, r - 1, L - 2)
        return t
    return g(0, len(s) - 1, L)

def pal_len_brute(s, L):
    return len({"".join(s[k] for k in idx) for idx in combinations(range(len(s)), L)
                if "".join(s[k] for k in idx) == "".join(s[k] for k in idx)[::-1]})

rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))
    assert interval_dp_while_ans(s) == count_brute(s)
    L = rng.randint(1, len(s))
    assert count_distinct_pal_len_ans(s, L) == pal_len_brute(s, L)
    assert sum(count_distinct_pal_len_ans(s, k) for k in range(1, len(s) + 1)) == count_brute(s)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "đếm palindrome phân biệt theo ký tự ngoài cùng":**

```python
@lru_cache(None)
def f(i, j):
    if i > j: return 0
    t = 0
    for x in alphabet:
        l, r = nxt[i][x], prv[j][x]        # x đầu tiên / cuối cùng trong [i, j]
        if l > j: continue
        t += 1 if l == r else 2 + f(l + 1, r - 1)
    return t % MOD
```

**Template DP đoạn (hai đầu bằng nhau `x`):** `0 x bên trong → 2·in + 2`; `1 x → 2·in + 1`; `≥ 2 x → 2·in - dp[low+1][high-1]`.

**Checklist trước khi submit:**

- [ ] Đếm **chuỗi phân biệt**, không đếm theo vị trí.
- [ ] Dùng `x` **đầu tiên** và **cuối cùng** trong đoạn.
- [ ] Ba trường hợp khi `s[i] == s[j]` (0 / 1 / ≥ 2 ký tự giống bên trong).
- [ ] Đoạn rỗng (`i > j`) = 0; cẩn thận `j == i + 1`.
- [ ] `% MOD` sau mỗi phép gán; phép trừ ổn trong Python.
- [ ] Đệ quy: tăng recursion limit (độ sâu ≤ n/2).
- [ ] Test `"a"`, `"aaaa"`, `"bccb"`, `"abcba"`.